# 🔍 Monitor de Vagas InHire - Versão Google Search

**Estratégia:** Como o InHire é um SPA (Single Page Application), vamos usar **Google Search** para encontrar vagas já indexadas.

**Vantagens:**
- ✅ Não precisa executar JavaScript
- ✅ Mais rápido (Google já indexou)
- ✅ Filtra por empresa + keywords em uma query
- ✅ Links diretos das vagas

**Técnicas:**
1. Google Dorking avançado
2. DuckDuckGo (backup - não bloqueia)
3. Extração de metadados dos resultados
4. Filtros inteligentes

---
## 1️⃣ Instalação de Dependências

In [ ]:
# Instalação
!pip install requests beautifulsoup4 pandas fake-useragent duckduckgo-search --quiet

print("✅ Dependências instaladas!")

---
## 2️⃣ Importações e Configurações

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime
from urllib.parse import quote_plus, urlparse
from fake_useragent import UserAgent
import warnings
warnings.filterwarnings('ignore')

ua = UserAgent()

def get_headers():
    return {
        'User-Agent': ua.random,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
    }

print("✅ Bibliotecas carregadas!")
print(f"📅 Execução: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

---
## 3️⃣ Configurações Personalizáveis

In [ ]:
# ========== EMPRESAS PARA MONITORAR ==========

COMPANIES = [
    "sympla",
    "agenciacriativa","agrosearch","alice","alun","amcom","auvotecnologia",
    "bankme","betaonline","brasilparalelo","ceisc","cerc","cielo","cora",
    "crown","db1","deloitte","digix","fitenergia","flash","flutterbrazil",
    "fretadao","gex","grupoescalar","gx2","infotecbrasil","kanastra","kpmg",
    "livemode","magalu","magazord","milvus","mjv","nomadglobal","olist",
    "openlabs","orizon","paytrack","pier","premiersoft","qive","radix",
    "residclub","sanar","sharepeoplehub","shareprime","supertex","sylvamo",
    "talentt","talentx","tripla","unimar","upflux","v360",
    "v4company","vitru","warren","zig","contabilizei","kiwify","loft",
    "nubank","creditas","ifood","stone","loggi"
]

# ========== KEYWORDS DE BUSCA ==========

# Cargos de interesse
KEYWORDS_CARGOS = [
    "analista de dados",
    "data analyst",
    "business analyst",
    "analista de negócios",
    "data engineer",
    "data scientist",
    "business intelligence",
]

# Modalidade
KEYWORDS_REMOTO = [
    "remoto",
    "remote",
    "home office",
]

# ========== CONFIGURAÇÕES DE BUSCA ==========

MAX_RESULTADOS_POR_QUERY = 20  # Máximo de resultados por busca
DELAY_ENTRE_QUERIES = 3  # Segundos entre buscas (evita bloqueio)
USAR_DUCKDUCKGO = True  # Tentar DuckDuckGo primeiro (mais confiável)

print(f"✅ Configurado para monitorar {len(COMPANIES)} empresas")
print(f"🔍 Buscando {len(KEYWORDS_CARGOS)} tipos de vagas")

---
## 4️⃣ Função: Buscar com DuckDuckGo (Recomendado)

**Por que DuckDuckGo?**
- ✅ Não bloqueia scraping facilmente
- ✅ Tem biblioteca Python oficial
- ✅ Mais confiável para automação

In [ ]:
def buscar_vagas_duckduckgo(query, max_results=20):
    """
    Busca vagas usando DuckDuckGo.
    
    Args:
        query (str): Query de busca
        max_results (int): Máximo de resultados
    
    Returns:
        list: Lista de dicionários com resultados
    """
    try:
        from duckduckgo_search import DDGS
        
        resultados = []
        
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)
            
            for result in results:
                # Extrai informações
                url = result.get('href', '')
                title = result.get('title', '')
                snippet = result.get('body', '')
                
                # Verifica se é realmente do InHire
                if 'inhire.app/vagas' in url:
                    # Extrai empresa do URL
                    match = re.search(r'https?://([a-zA-Z0-9-]+)\.inhire\.app', url)
                    empresa = match.group(1).upper() if match else "DESCONHECIDA"
                    
                    resultados.append({
                        'empresa': empresa,
                        'titulo': title,
                        'link': url,
                        'descricao': snippet,
                        'fonte': 'DuckDuckGo'
                    })
        
        return resultados
        
    except ImportError:
        print("⚠️  duckduckgo-search não instalado")
        return []
    except Exception as e:
        print(f"⚠️  Erro DuckDuckGo: {str(e)[:50]}")
        return []

print("✅ Função DuckDuckGo criada!")

---
## 5️⃣ Função: Buscar com Google (Backup)

**Limitações:**
- ⚠️ Google pode bloquear após muitas buscas
- ⚠️ Menos confiável para automação
- ✅ Mas tem mais resultados quando funciona

In [ ]:
def buscar_vagas_google(query, max_results=20):
    """
    Busca vagas usando Google (pode ser bloqueado).
    
    Args:
        query (str): Query de busca
        max_results (int): Máximo de resultados
    
    Returns:
        list: Lista de dicionários com resultados
    """
    resultados = []
    
    try:
        # Codifica query para URL
        query_encoded = quote_plus(query)
        
        # URL do Google Search
        google_url = f"https://www.google.com/search?q={query_encoded}&num={min(max_results, 50)}"
        
        # Requisição
        response = requests.get(google_url, headers=get_headers(), timeout=10)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extrai resultados (Google usa classes dinâmicas, então tentamos vários seletores)
            for result_div in soup.find_all('div', class_=['g', 'Gx5Zad']):
                # Procura link
                link_tag = result_div.find('a')
                if not link_tag:
                    continue
                
                url = link_tag.get('href', '')
                
                # Verifica se é do InHire
                if 'inhire.app/vagas' not in url:
                    continue
                
                # Extrai título
                title_tag = result_div.find(['h3', 'h2'])
                title = title_tag.get_text(strip=True) if title_tag else "Sem título"
                
                # Extrai snippet
                snippet_tag = result_div.find(['div', 'span'], class_=re.compile('VwiC3b|s3v9rd'))
                snippet = snippet_tag.get_text(strip=True) if snippet_tag else ""
                
                # Extrai empresa
                match = re.search(r'https?://([a-zA-Z0-9-]+)\.inhire\.app', url)
                empresa = match.group(1).upper() if match else "DESCONHECIDA"
                
                resultados.append({
                    'empresa': empresa,
                    'titulo': title,
                    'link': url,
                    'descricao': snippet,
                    'fonte': 'Google'
                })
        
        else:
            print(f"⚠️  Google retornou status {response.status_code}")
    
    except Exception as e:
        print(f"⚠️  Erro Google: {str(e)[:50]}")
    
    return resultados

print("✅ Função Google criada!")

---
## 6️⃣ Função: Construir Queries Otimizadas

**Estratégia:** Criar queries que maximizam resultados relevantes.

In [ ]:
def construir_queries(empresas=None, keywords_cargos=None, keywords_remoto=None):
    """
    Constrói queries otimizadas para busca.
    
    Args:
        empresas (list): Lista de empresas (None = todas)
        keywords_cargos (list): Keywords de cargos
        keywords_remoto (list): Keywords de modalidade
    
    Returns:
        list: Lista de queries
    """
    empresas = empresas or COMPANIES
    keywords_cargos = keywords_cargos or KEYWORDS_CARGOS
    keywords_remoto = keywords_remoto or KEYWORDS_REMOTO
    
    queries = []
    
    # ESTRATÉGIA 1: Query geral (todos os cargos + remoto)
    cargos_or = ' OR '.join([f'"{cargo}"' for cargo in keywords_cargos])
    remoto_or = ' OR '.join([f'"{r}"' for r in keywords_remoto])
    
    query_geral = f'site:inhire.app/vagas ({cargos_or}) ({remoto_or})'
    queries.append(('GERAL', query_geral))
    
    # ESTRATÉGIA 2: Por empresa (para empresas prioritárias)
    empresas_prioritarias = empresas[:10]  # Primeiras 10
    
    for empresa in empresas_prioritarias:
        query_empresa = f'site:{empresa}.inhire.app/vagas ({cargos_or})'
        queries.append((empresa.upper(), query_empresa))
    
    return queries

# Teste
queries_teste = construir_queries(empresas=['sympla'], keywords_cargos=['analista de dados'])
print("✅ Função de queries criada!")
print(f"\nExemplo de query:")
print(f"  {queries_teste[0][1]}")

---
## 7️⃣ Filtros Inteligentes

**Filtrar resultados** para garantir relevância.

In [ ]:
def filtrar_vagas(vagas):
    """
    Aplica filtros de relevância nas vagas.
    
    Args:
        vagas (list): Lista de vagas brutas
    
    Returns:
        list: Vagas filtradas
    """
    vagas_filtradas = []
    urls_vistas = set()  # Evita duplicatas
    
    for vaga in vagas:
        url = vaga['link']
        
        # Remove duplicatas
        if url in urls_vistas:
            continue
        
        urls_vistas.add(url)
        
        # Texto para análise
        texto = (vaga['titulo'] + ' ' + vaga['descricao']).lower()
        
        # Filtro 1: Verifica se tem keyword de cargo
        tem_cargo = any(cargo.lower() in texto for cargo in KEYWORDS_CARGOS)
        
        # Filtro 2: Verifica se tem keyword de remoto
        tem_remoto = any(remoto.lower() in texto for remoto in KEYWORDS_REMOTO)
        
        # Filtro 3: Não é "não remoto" ou "presencial"
        falso_positivo = any(palavra in texto for palavra in ['não remoto', 'presencial', 'híbrido'])
        
        # Aplica filtros
        if tem_cargo and tem_remoto and not falso_positivo:
            vagas_filtradas.append(vaga)
        elif tem_cargo:  # Se tem cargo mas não mencionou remoto, inclui (pode estar no link)
            vagas_filtradas.append(vaga)
    
    return vagas_filtradas

print("✅ Função de filtros criada!")

---
## 8️⃣ EXECUÇÃO PRINCIPAL

**Fluxo:**
1. Constrói queries otimizadas
2. Busca com DuckDuckGo (ou Google como backup)
3. Filtra resultados
4. Gera DataFrame

In [ ]:
print("🚀 INICIANDO BUSCA DE VAGAS\n")
print("=" * 70)

# Constrói queries
print("\n📋 CONSTRUINDO QUERIES...")
queries = construir_queries()
print(f"✅ {len(queries)} queries criadas\n")

# Armazena todos os resultados
todas_vagas = []

# Executa buscas
print("=" * 70)
print("🔍 EXECUTANDO BUSCAS")
print("=" * 70)

for i, (nome, query) in enumerate(queries, 1):
    print(f"\n[{i}/{len(queries)}] {nome}")
    print(f"Query: {query[:80]}..." if len(query) > 80 else f"Query: {query}")
    
    # Tenta DuckDuckGo primeiro
    if USAR_DUCKDUCKGO:
        resultados = buscar_vagas_duckduckgo(query, MAX_RESULTADOS_POR_QUERY)
        
        if resultados:
            print(f"  ✅ DuckDuckGo: {len(resultados)} resultados")
            todas_vagas.extend(resultados)
        else:
            # Fallback para Google
            print(f"  ⚠️  DuckDuckGo falhou, tentando Google...")
            resultados = buscar_vagas_google(query, MAX_RESULTADOS_POR_QUERY)
            
            if resultados:
                print(f"  ✅ Google: {len(resultados)} resultados")
                todas_vagas.extend(resultados)
            else:
                print(f"  ❌ Nenhum resultado")
    else:
        # Usa apenas Google
        resultados = buscar_vagas_google(query, MAX_RESULTADOS_POR_QUERY)
        
        if resultados:
            print(f"  ✅ {len(resultados)} resultados")
            todas_vagas.extend(resultados)
        else:
            print(f"  ❌ Nenhum resultado")
    
    # Delay entre queries
    if i < len(queries):
        time.sleep(DELAY_ENTRE_QUERIES)

print("\n" + "=" * 70)
print(f"📊 TOTAL DE RESULTADOS BRUTOS: {len(todas_vagas)}")
print("=" * 70)

# Filtra resultados
print("\n🎯 APLICANDO FILTROS...")
vagas_filtradas = filtrar_vagas(todas_vagas)
print(f"✅ VAGAS RELEVANTES: {len(vagas_filtradas)}")

print("\n" + "=" * 70)
print("✅ BUSCA CONCLUÍDA!")
print("=" * 70)

---
## 9️⃣ Visualização dos Resultados

In [ ]:
if vagas_filtradas:
    # Cria DataFrame
    df_vagas = pd.DataFrame(vagas_filtradas)
    
    # Remove coluna de descrição (muito longa para visualização)
    df_display = df_vagas[['empresa', 'titulo', 'link', 'fonte']].copy()
    
    # Renomeia colunas
    df_display.columns = ['Empresa', 'Título da Vaga', 'Link', 'Fonte']
    
    # Adiciona índice
    df_display.index = range(1, len(df_display) + 1)
    
    print("\n" + "=" * 80)
    print(f"📊 {len(df_display)} VAGAS ENCONTRADAS")
    print("=" * 80 + "\n")
    
    # Exibe
    display(df_display)
    
    # Estatísticas
    print("\n" + "=" * 80)
    print("📈 ESTATÍSTICAS")
    print("=" * 80)
    print(f"\n🏢 Empresas com vagas: {df_display['Empresa'].nunique()}")
    print(f"\nTop 10 empresas:")
    print(df_display['Empresa'].value_counts().head(10))
    
    print(f"\n📊 Fontes:")
    print(df_display['Fonte'].value_counts())
    
else:
    print("\n⚠️  Nenhuma vaga encontrada.")
    print("\nSugestões:")
    print("  - Aumente MAX_RESULTADOS_POR_QUERY")
    print("  - Reduza filtros (aceite híbrido)")
    print("  - Tente queries mais genéricas")
    df_vagas = pd.DataFrame()

---
## 🔟 Exportação

In [ ]:
if not df_vagas.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # CSV
    csv_file = f'vagas_inhire_{timestamp}.csv'
    df_vagas.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✅ CSV salvo: {csv_file}")
    
    # Excel
    try:
        excel_file = f'vagas_inhire_{timestamp}.xlsx'
        df_vagas.to_excel(excel_file, index=False, engine='openpyxl')
        print(f"✅ Excel salvo: {excel_file}")
    except:
        print("⚠️  Excel não salvo (instale: pip install openpyxl)")
else:
    print("⚠️  Nenhum dado para exportar")

---
## 📚 Próximos Passos

### Melhorias Possíveis:

1. **Visitar links individuais** para extrair mais detalhes (salário, requisitos)
2. **Banco de dados** para não duplicar alertas
3. **Telegram** para notificações
4. **Agendamento** para rodar diariamente

### Limitações:

- ⚠️ Google pode bloquear após muitas buscas (use DuckDuckGo)
- ⚠️ Resultados dependem do índice do buscador
- ⚠️ Vagas muito novas podem não estar indexadas

### Vantagens:

- ✅ Não precisa Selenium (mais rápido)
- ✅ Funciona com SPAs
- ✅ Filtra múltiplas empresas em uma busca
- ✅ Mais confiável que scraping direto